# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [2]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from anthropic import Anthropic

In [5]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('ANTHROPIC_API_KEY')

if api_key and api_key.startswith('sk-ant-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'claude-sonnet-4-6'
client = Anthropic(api_key=api_key)

API key looks good so far


In [4]:
links = fetch_website_links("https://edwarddonner.com")
links

['https://edwarddonner.com/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [24]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page (Sobre Nós in pt-br), Quem Somos, or a Company page, or Careers/Jobs pages (Trabalhe Conosco in pt-br), blog, blog posts, Nossa Equipe.
Don't use Markdown.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [7]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [8]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

https://edwarddonner.com/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.

In [27]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = client.messages.create(
        model=MODEL, 
        max_tokens=1024, 
        system=link_system_prompt,
        messages=[{"role": "user", "content": get_links_user_prompt(url)}],
    )
    result = response.content[0].text
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [28]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling claude-sonnet-4-6
Found 8 relevant links


{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'blog posts', 'url': 'https://edwarddonner.com/posts/'},
  {'type': 'curriculum page', 'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'company page',
   'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'},
  {'type': 'blog post',
   'url': 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/'},
  {'type': 'blog post',
   'url': 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/'},
  {'type': 'blog post',
   'url': 'https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-aws-at-scale/'},
  {'type': 'blog post',
   'url': 'https://edwarddonner.com/2025/05/28/connecting-my-courses-become-an-llm-expert-and-leader/'}]}

In [29]:
select_relevant_links("https://www.anageimoveis.com.br")

Selecting relevant links for https://www.anageimoveis.com.br by calling claude-sonnet-4-6
Found 0 relevant links


{'links': []}

In [30]:
select_relevant_links("https://nozzimobiliaria.com.br")

Selecting relevant links for https://nozzimobiliaria.com.br by calling claude-sonnet-4-6
Found 3 relevant links


{'links': [{'type': 'about page',
   'url': 'https://nozzimobiliaria.com.br/sobre'},
  {'type': 'contact page', 'url': 'https://nozzimobiliaria.com.br/contato'},
  {'type': 'advertise page', 'url': 'https://nozzimobiliaria.com.br/anuncie'}]}

In [31]:
select_relevant_links("https://www.principeimoveisjoinville.com.br")

Selecting relevant links for https://www.principeimoveisjoinville.com.br by calling claude-sonnet-4-6
Found 5 relevant links


{'links': [{'type': 'about page',
   'url': 'https://www.principeimoveisjoinville.com.br/sobre'},
  {'type': 'careers page',
   'url': 'https://www.principeimoveisjoinville.com.br/trabalhe-conosco'},
  {'type': 'team page',
   'url': 'https://www.principeimoveisjoinville.com.br/nossa-equipe'},
  {'type': 'blog', 'url': 'https://www.principeimoveisjoinville.com.br/blog'},
  {'type': 'contact page',
   'url': 'https://www.principeimoveisjoinville.com.br/contato'}]}

In [32]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling claude-sonnet-4-6
Found 20 relevant links


{'links': [{'type': 'about page', 'url': 'https://huggingface.co/huggingface'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'},
  {'type': 'blog', 'url': 'https://huggingface.co/blog'},
  {'type': 'learn page', 'url': 'https://huggingface.co/learn'},
  {'type': 'brand page', 'url': 'https://huggingface.co/brand'},
  {'type': 'models page', 'url': 'https://huggingface.co/models'},
  {'type': 'datasets page', 'url': 'https://huggingface.co/datasets'},
  {'type': 'spaces page', 'url': 'https://huggingface.co/spaces'},
  {'type': 'docs page', 'url': 'https://huggingface.co/docs'},
  {'type': 'changelog page', 'url': 'https://huggingface.co/changelog'},
  {'type': 'partner page', 'url': 'https://huggingface.co/allenai'},
  {'type': 'partner page', 'url': 'https://huggingface.co/facebook'},
  {'type': 'partner page', 'ur

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [33]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [34]:
print(fetch_page_and_all_relevant_links("https://www.principeimoveisjoinville.com.br"))

Selecting relevant links for https://www.principeimoveisjoinville.com.br by calling claude-sonnet-4-6
Found 5 relevant links
## Landing Page:


    A Melhor Imobiliária Em Joinville - Príncipe Imóveis 

✖
Imóveis
Imóveis à venda
Imóveis para alugar
Condomínios
Anunciar seu imóvel
Favoritos
Cliente
Área do Inquilino
Área do proprietário
Ouvidoria
A Príncipe Imóveis
Quem Somos
Fale Conosco
Trabalhe Conosco
Nossa equipe
Documentos
Blog
Política de Privacidade
Contatos
(47) 3227-7747
vendas@principeimoveisjoinville.com.br
CRECI: 6333J
(47) 3227-7747
ÁREA DO PROPRIETÁRIO
ÁREA DO INQUILINO
COMPRAR
ALUGAR
ANUNCIAR MEU IMÓVEL
ÁREA DO CLIENTE
×
Home
Comprar
Alugar
Anunciar seu imóvel
Cadastrar seu
                                    imóvel
Condomínios
Ouvidoria
Favoritos
0
Quem somos
Área do cliente
Área do
                                    locatário
Área do
                                    proprietário
Blog
Contato
vendas@principeimoveisjoinville.com.br
(47) 3227-7747
(47) 9 9980-1754 -
 

In [35]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
Respond in Brazilian Portuguese (pt-br).
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [36]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks. Respond in Brazilian Portuguese (pt-br).\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [39]:
get_brochure_user_prompt("Príncipe Imóveis", "https://www.principeimoveisjoinville.com.br")

Selecting relevant links for https://www.principeimoveisjoinville.com.br by calling claude-sonnet-4-6
Found 5 relevant links


'\nYou are looking at a company called: Príncipe Imóveis\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks. Respond in Brazilian Portuguese (pt-br).\n\n\n## Landing Page:\n\n\n    A Melhor Imobiliária Em Joinville - Príncipe Imóveis \n\n✖\nImóveis\nImóveis à venda\nImóveis para alugar\nCondomínios\nAnunciar seu imóvel\nFavoritos\nCliente\nÁrea do Inquilino\nÁrea do proprietário\nOuvidoria\nA Príncipe Imóveis\nQuem Somos\nFale Conosco\nTrabalhe Conosco\nNossa equipe\nDocumentos\nBlog\nPolítica de Privacidade\nContatos\n(47) 3227-7747\nvendas@principeimoveisjoinville.com.br\nCRECI: 6333J\n(47) 3227-7747\nÁREA DO PROPRIETÁRIO\nÁREA DO INQUILINO\nCOMPRAR\nALUGAR\nANUNCIAR MEU IMÓVEL\nÁREA DO CLIENTE\n×\nHome\nComprar\nAlugar\nAnunciar seu imóvel\nCadastrar seu\n                                    imóvel\nCondomínios\nOuvidoria\nFavoritos\n0\nQuem somos\nÁrea do cliente\nÁrea do

In [40]:
def create_brochure(company_name, url):
    response = client.messages.create(model=MODEL, max_tokens=1024, system=brochure_system_prompt, messages=[{"role": "user", "content": get_brochure_user_prompt(company_name, url)}])
    result = response.content[0].text
    display(Markdown(result))

In [41]:
create_brochure("Príncipe Imóveis", "https://www.principeimoveisjoinville.com.br")

Selecting relevant links for https://www.principeimoveisjoinville.com.br by calling claude-sonnet-4-6
Found 5 relevant links


# Príncipe Imóveis — Sua Imobiliária de Referência em Joinville

---

## Quem Somos

A **Príncipe Imóveis** é uma imobiliária de referência em Joinville, SC, comprometida em ser muito mais do que uma simples corretora de imóveis. Somos a sua parceira de confiança para encontrar o lar dos seus sonhos ou realizar um investimento seguro e tranquilo.

Com uma equipe experiente e atendimento personalizado, acompanhamos nossos clientes em **todas as etapas do processo**, cuidando de cada detalhe para proporcionar uma experiência sem complicações.

---

## Nossos Diferenciais

- 🔒 **Segurança Jurídica** — Rigoroso processo de análise para garantir transações seguras e transparentes
- 📋 **Informações Completas** — Fotos, vídeos, endereço e todas as características importantes de cada imóvel
- ⚙️ **Processos Otimizados** — Cuidamos de toda a burocracia para que você foque no que realmente importa
- 🤝 **Atendimento Personalizado** — Uma equipe dedicada a entender as suas necessidades

---

## O Que Oferecemos

### Para Compradores e Locatários
- Imóveis à venda e para alugar em Joinville
- Apartamentos, casas e lançamentos em diversos bairros
- Filtros de busca por tipo, bairro, número de quartos e valor
- Imóveis com piscina, varanda, armários e muito mais

### Para Proprietários
- Anúncio do seu imóvel conectado a clientes realmente interessados
- Área exclusiva do proprietário para acompanhamento
- Gestão completa do processo de locação e venda

### Para Inquilinos
- Área exclusiva do locatário
- Canal de ouvidoria para suporte e atendimento

---

## Bairros Atendidos em Joinville

América, Glória, Santo Antônio, Costa e Silva, Anita Garibaldi, Atiradores, Floresta, Guanabara, Iririú, Saguaçu, Bom Retiro, Boa Vista, Bucarein, Centro, entre outros.

---

## Trabalhe Conosco

A Príncipe Imóveis está sempre em busca de talentos apaixonados pelo mercado imobiliário. Se você deseja fazer parte de uma equipe experiente e de referência em Joinville, entre em contato conosco!

📧 **vendas@principeimoveisjoinville.com.br**

---

## Fale Conosco

| Canal | Contato |
|---|---|
| 📞 Telefone Geral | (47) 3227-7747 |
| 📱 WhatsApp Locação | (47) 9 9980-1754 |
| 📱 WhatsApp Venda | (47) 9 9794-5042 |
| 📧 E-mail | vendas@principeimoveisjoinville.com.br |
| 🏢 CRECI | 6333J |

---

*Príncipe Imóveis — Segurança, confiança e tranquilidade no mercado imobiliário de Joinville.*

In [43]:
create_brochure("NOZZ Imobiliária", "https://nozzimobiliaria.com.br")

Selecting relevant links for https://nozzimobiliaria.com.br by calling claude-sonnet-4-6
Found 3 relevant links


# NOZZ Curadoria Imobiliária

### Encontre o imóvel dos seus sonhos com quem entende de qualidade

---

## Quem Somos

A **NOZZ Curadoria Imobiliária** nasceu de uma inspiração simples, mas profundamente simbólica: a estrutura de uma noz. Assim como a noz é forte por fora e guarda algo valioso em seu interior, a NOZZ acredita que a verdadeira beleza de um imóvel está na qualidade do que ele oferece — e não apenas na aparência superficial.

O nome também dialoga com a fonética de **"nós"**, reforçando o espírito colaborativo e humano da empresa. Acreditamos que o que permanece e faz sentido se fortalece, tornando a equipe mais coesa, consciente e sólida a cada dia.

> *"A forma existe para servir à experiência."*

---

## Nossa Missão, Visão e Valores

- **Missão:** Conectar pessoas aos imóveis ideais com curadoria, cuidado e transparência.
- **Visão:** Ser referência em curadoria imobiliária em Santa Catarina, priorizando a experiência do cliente acima de tudo.
- **Valores:** Solidez, coesão, autenticidade e compromisso com a qualidade.

---

## Nossos Serviços

### 🏠 Compra e Venda de Imóveis
Oferecemos um portfólio criteriosamente selecionado de imóveis residenciais e comerciais, desde apartamentos compactos até mansões de alto padrão.

### 📢 Anuncie seu Imóvel
Proprietários podem anunciar seus imóveis de forma simples e rápida. Nossa equipe cuida de todo o processo, incluindo planejamento fotográfico e divulgação qualificada.

---

## Destaques do Portfólio

| Código | Tipo | Localização | Área Privativa | Preço |
|--------|------|-------------|---------------|-------|
| NZ1199 | Casa em Condomínio | América, Joinville/SC | 755 m² | R$ 9.500.000,00 |
| NZ1007 | Casa | América, Joinville/SC | 382 m² | R$ 6.990.000,00 |
| NZ1024 | Casa | Floresta, Joinville/SC | 212 m² | R$ 950.000,00 |
| NZ1155 | Apartamento | Saguaçu, Joinville/SC | 68 m² | R$ 625.000,00 |
| NZL1008 | Apartamento com Vista Mar | Balneário Piçarras/SC | 74 m² | R$ 799.000,00 |

---

## Certificações

✅ **CRECI:** 10395
✅ **Selo de Tecnologia Loft** — imobiliária certificada em inovação e tecnologia do setor

---

## Onde Estamos

📍 Rua Coelho Neto, 446, Sala 1 — Santo Antônio, Joinville/SC — CEP 89218-015

---

## Fale Conosco

📞 **(47) 99616-9162**
📧 **rafael@nozzimobiliaria.com.br**
🌐 [nozzimobiliaria.com.br](http://nozzimobiliaria.com.br)

---

*NOZZ Curadoria Imobiliária — Solidez, qualidade e cuidado em cada transação.*

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [44]:
def stream_brochure(company_name, url):
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    with client.messages.stream(
        max_tokens=1024,
        model=MODEL,
        system=brochure_system_prompt,
       messages=[{"role": "user", "content": get_brochure_user_prompt(company_name, url)}]
    ) as stream:
        for text in stream.text_stream:
            response += text or ''
            update_display(Markdown(response), display_id=display_handle.display_id)
            #print(text, end="", flush=True)



    
    #for chunk in stream:
        

In [45]:
stream_brochure("NOZZ Imobiliária", "https://nozzimobiliaria.com.br")

# NOZZ Curadoria Imobiliária

### Encontre o imóvel que guarda o que há de mais valioso para você

---

## Quem Somos

A **NOZZ Curadoria Imobiliária** é uma empresa especializada na intermediação de imóveis, com sede em **Joinville, Santa Catarina**. O nome NOZZ é inspirado na estrutura de uma noz — um elemento simples da natureza, mas profundamente simbólico: forte por fora, valioso por dentro. Assim como a noz, a empresa acredita que a verdadeira qualidade está no que se guarda com cuidado.

O nome também dialoga com a fonética de **"nós"** — e esse duplo sentido é intencional. A NOZZ é construída sobre relações humanas, conexões genuínas e um senso de pertencimento coletivo.

> *"A forma existe para servir à experiência."*

---

## Nossa Missão, Visão e Valores

- **Missão:** Oferecer uma curadoria imobiliária cuidadosa, conectando pessoas aos imóveis que realmente fazem sentido para suas vidas.
- **Visão:** Ser referência em qualidade e confiança no mercado imobiliário de Joinville e região.
- **Valores:** Solidez, transparência, coesão e evolução constante. Acreditamos que o que permanece se fortalece — e que cada transação deve ser uma experiência positiva e significativa.

---

## Nossos Diferenciais

- ✅ **CRECI 10395** — empresa regularizada e certificada
- 🏅 **Certificação Loft** — Selo de Tecnologia Loft, reconhecimento de excelência em inovação no setor imobiliário
- 🏡 Curadoria personalizada de imóveis residenciais e comerciais
- 📍 Foco em Joinville/SC e região
- 📸 Suporte completo para anunciantes, incluindo planejamento fotográfico profissional

---

## Imóveis em Destaque

Confira alguns dos imóveis disponíveis em nosso portfólio:

| Código | Tipo | Bairro | Valor |
|--------|------|--------|-------|
| NZ1199 | Casa em Condomínio | América, Joinville/SC | R$ 9.500.000,00 |
| NZ1007 | Casa | América, Joinville/SC | R$ 6.990.000,00 |
| NZ1155 | Apartamento | Saguaçu, Joinville/SC | R$ 625.000,00 |
| NZ1024 | Casa | Floresta, Joinville/SC | R$ 950.000,00 |
| NZ1206 | Apartamento | Atiradores, Joinville/SC | R$ 850.000,00 |

Temos opções para todos os perfis — de apartamentos compactos a mansões em condomínio fechado.

---

## Anuncie seu Imóvel

Quer vender ou alugar seu imóvel? A NOZZ oferece um processo simples e completo:

1. Preencha o formulário online com as informações do imóvel
2. Envie fotos iniciais (sem preocupação com qualidade — cuidamos disso!)
3. Nossa equipe entra em contato e planeja as fotografias profissionais do anúncio
4. Seu imóvel é apresentado com curadoria e qualidade para os compradores certos

---

## Fale Conosco

📍 **Endereço:** Rua Coelho Neto, 446, Sala 1 — Santo Antônio, Joinville/SC — CEP 89218-015

📞 **Telefone/WhatsApp:** (47) 99616-9162

📧 **E-mail:** rafael@nozzimobiliaria.com.br

🌐 **Site:** nozzim

Selecting relevant links for https://nozzimobiliaria.com.br by calling claude-sonnet-4-6
Found 3 relevant links


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>